# Quy trình hệ thống chạy — từng bước, chỉnh được, thực nghiệm trực tiếp

Notebook `01` giải thích **mô hình** (dữ liệu, đồ thị, huấn luyện). Notebook này chạy **cả hệ thống** như khi người dùng
gọi `assess("…", L_peak=150)`, nhưng **tách từng bước ra** để bạn xem đầu vào/đầu ra của mỗi bước và **đổi tham số
rồi chạy lại**.

```
 Yêu cầu (tiếng Việt/Anh)
   │
   ▼  BƯỚC 1  ParserAgent ── luật (từ khoá) → LLM → 6 guard (G1–G6)  ⇒ archetype, Δ, dịch vụ
   │
   ▼  BƯỚC 2  ArchitectureAgent ── quét đồ thị phụ thuộc                ⇒ vùng ảnh hưởng
   │
   ├─▶ BƯỚC 3  TẦNG 1 — CapacityAgent (Global Causal DAG)            ⇒ CPU/mem/độ trễ dự kiến, biên dự phòng
   │
   └─▶ BƯỚC 4  TẦNG 2 — bộ dự đoán khả thi (u_s vs u*, cổng throttling) ⇒ FEASIBLE / MARGINAL / INFEASIBLE, điểm gãy R*
```

**Cách dùng:**
1. Chọn kernel **`.venv`**, *Run All* một lần (~1 phút) để thấy cấu hình mặc định.
2. Sửa **ô CẤU HÌNH** (mục 0), rồi *Run All* lại — hoặc chạy lại từ ô cấu hình trở xuống.
3. Muốn thử một ý tưởng mô hình: dùng **mục 6 — Bàn thực nghiệm**, kết quả tự vào **bảng xếp hạng**.

**Bảo đảm an toàn:** mọi thay đổi ở đây chỉ nằm **trong bộ nhớ** của notebook — không ghi file nào, không đụng file đóng
băng, không sửa `src/`. Tắt kernel là mọi thứ về như cũ. Đừng chạy khi đang đo ramp (làm nhiễu phép đo).

> **Phát triển ≠ tiến cứu.** Mọi so sánh ở mục 6 dùng dữ liệu đáp án **đã xem**. Một biến thể thắng ở đây mới chỉ là
> ứng viên; muốn gọi là tiến cứu phải đưa vào `src/` (chế độ mới), đóng băng bằng script, commit **và push**, rồi đo
> trên tính năng **mới** (mục 7).

In [ ]:
import glob, json, os, sys, re, copy, contextlib, io, warnings
from dataclasses import asdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, networkx as nx

ROOT = Path.cwd()
while not (ROOT / 'src').exists():
    ROOT = ROOT.parent
for p in ['src', 'src/agents', 'src/scm', 'experiments/collect', 'experiments/feasibility']:
    if str(ROOT / p) not in sys.path:
        sys.path.insert(0, str(ROOT / p))
warnings.filterwarnings('ignore'); os.environ.setdefault('TQDM_DISABLE', '1')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
FROZEN, RAW = ROOT / 'data/processed/frozen', ROOT / 'data/raw'

@contextlib.contextmanager
def quiet():
    """an log dai (dowhy, agent). Muon xem log: thay `with quiet():` bang `if True:`"""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

import feasibility_predictor as FP
import parser_agent as PA
import capacity_agent as CA
import load_sweep_collect as HARNESS
from request_router import SOCKSHOP_CALL_CHAINS, classify_request
from architecture_agent import ArchitectureAgent
try:
    display
except NameError:
    display = print
print('goc repo:', ROOT)

## 0. ⚙️ CẤU HÌNH — mọi núm chỉnh nằm ở đây

Giá trị mặc định = **đúng cấu hình hệ thống đang dùng** (bản đóng băng). Sửa một giá trị rồi chạy lại từ ô này trở xuống.

In [ ]:
CAU_HINH = dict(
    # ---------------- ĐẦU VÀO ----------------
    YEU_CAU = "Khách hàng muốn đăng nhập nhanh chỉ bằng một nút",
    L_PEAK  = 150,             # tải đỉnh kỳ vọng (req/s). None = chỉ chạy Tầng 1 (dự báo), không phán quyết
    K       = 'probe',         # bội số gọi k: None (P2, giả định k=1) | 'probe' (lấy từ file probe theo archetype)
                               #   | 'probe:login' (của một tính năng cụ thể) | dict vd {'user': 1, 'carts': 1}

    # ---------------- LLM (BƯỚC 1) ----------------
    LLM = 'offline',           # 'offline'  : không gọi LLM -> Parser lùi về luật (giống máy hiện tại, chưa có API key)
                               # 'gia_lap'  : LLM giả trả đúng JSON ở LLM_GIA_LAP -> thử guard G1–G6 với đầu ra xấu
                               # tên backend: vd 'gemini-3.5-flash-lite' (cần API key trong .env; TỐN PHÍ)
    LLM_GIA_LAP = {            # chỉ dùng khi LLM='gia_lap'. Thử: injection_service='orders' (G1),
        'is_customer_facing_feature': True,     #   adjustment=40 (G3), core_services=['redis'] (G4),
        'is_customer_facing_confidence': 'HIGH',#   is_customer_facing_feature=False (G6)
        'request_type': 'LOGIN', 'core_services': ['front-end', 'user'],
        'injection_service': 'front-end', 'adjustment': 5.0,
        'reasoning': 'LLM gia lap', 'confidence': 'HIGH'},

    # ---------------- GUARD CỦA PARSER ----------------
    MAX_ADJUSTMENT_PCT   = 10.0,   # G3: LLM chỉ được lệch ±x% so với anchor của archetype
    MIN_DELTA_PCT        = 5.0,    # G2: sàn Δ
    MAX_DELTA_PCT        = 50.0,   # G2: trần Δ
    SIMILARITY_THRESHOLD = 0.6,    # G5: dưới ngưỡng -> ép adjustment = 0
    TAU_PASS             = 0.5343, # G6: điểm scope-gate < TAU_PASS -> cho qua
    TAU_REFUSE           = 0.6907, # G6: >= TAU_REFUSE -> từ chối; ở giữa -> cần người xem

    # ---------------- TẦNG 1: GLOBAL CAUSAL DAG (BƯỚC 3) ----------------
    MAU_CANH_HOC = ['cpu_backpressure', 'latency_backprop_causil'],   # bỏ bớt để xem DAG không có cạnh học
    DELTA_TANG1  = None,       # None = dùng Δ của Parser; hoặc một số (%) để ép

    # ---------------- TẦNG 2: BỘ DỰ ĐOÁN KHẢ THI (BƯỚC 4) ----------------
    CHE_DO        = 'P2',          # 'P0' | 'P1' | 'P2'   (P3 = P2 + K đo được)
    CHAIN         = 'taxonomy',    # 'taxonomy' (như bản đóng băng) | 'probe' (chain đo được) | list tuỳ ý
    U_STAR        = 'dong_bang',   # 'dong_bang' (0,8783) | 'phien_tien_cuu' (từ ramp base SS-PROSP2) | một số
    U_MARGIN      = 0.10,          # MARGINAL khi max_u trong 10% dưới u*
    TRAN_CPU      = 'RE2',         # tên cấu hình trong deploy/sockshop/limits.json, hoặc dict {service: core}
    TRAIN_MAX_RPS = 150,           # chỉ dùng dữ liệu SS-TRAIN có tải <= giá trị này để fit cơ chế
    THAM_SO_P2    = 'dong_bang',   # 'dong_bang' | dict {'c': {...}, 'x': ...}
    QUOTA_MIN_CORES = 0.5,         # cổng throttling: node nghẽn dưới ngưỡng này -> UNDECIDED
    N_BOOTSTRAP   = 100,           # số lần bootstrap cho khoảng tin cậy điểm gãy (0 = tắt)
)
CH = CAU_HINH
print('cau hinh da nap:', {k: v for k, v in CH.items() if k != 'LLM_GIA_LAP'})

In [ ]:
# --- ap cau hinh vao cac module (CHI trong bo nho) ---
for k in ['MAX_ADJUSTMENT_PCT', 'MIN_DELTA_PCT', 'MAX_DELTA_PCT', 'SIMILARITY_THRESHOLD']:
    setattr(PA, k, float(CH[k]))
PA._SCOPE_GATE_CONFORMAL_TAU_PASS, PA._SCOPE_GATE_CONFORMAL_TAU_REFUSE = float(CH['TAU_PASS']), float(CH['TAU_REFUSE'])

class _Resp:
    def __init__(self, content): self.content = content

class LLMGhiLai:
    """boc mot LLM bat ky, GHI LAI prompt gui di va cau tra loi nhan ve de xem o buoc 1"""
    def __init__(self, inner): self.inner, self.nhat_ky = inner, []
    def invoke(self, messages):
        prompt = '\n'.join(getattr(m, 'content', str(m)) for m in messages)
        try:
            r = self.inner.invoke(messages); out = getattr(r, 'content', str(r))
        except Exception as e:
            out = f'<LOI: {e}>'
        self.nhat_ky.append({'prompt': prompt, 'tra_loi': out})
        return _Resp(out)

class LLMOffline:
    """giong he thong khi chua co API key: tra ve van ban khong phai JSON -> Parser lui ve luat"""
    def invoke(self, messages): return _Resp('(offline: khong co LLM)')

class LLMGiaLap:
    def __init__(self, data): self.data = data
    def invoke(self, messages): return _Resp(json.dumps(self.data, ensure_ascii=False))

if CH['LLM'] == 'offline':
    _inner = LLMOffline()
elif CH['LLM'] == 'gia_lap':
    _inner = LLMGiaLap(CH['LLM_GIA_LAP'])
else:
    from llm_backends import get_llm
    _inner = get_llm(CH['LLM'])
LLM = LLMGhiLai(_inner)
arch = ArchitectureAgent(str(ROOT / 'src/graph/sockshop_agent_graph.json'))
parser = PA.ParserAgent(llm=LLM, arch_agent=arch)
print('LLM:', CH['LLM'], '| gateway cua he:', sorted(parser._gateways))

## 1. BƯỚC 1 — ParserAgent: yêu cầu → can thiệp có ràng buộc

LLM **không bao giờ** được tự quyết con số cuối. Nó chỉ *đề xuất*; sáu guard xác định (G1–G6) chiếu đề xuất về vùng hợp lệ.
Vì vậy các bảo đảm cấu trúc đúng với **mọi** đầu ra của LLM — kể cả đầu ra xấu (thử bằng `LLM='gia_lap'`).

### 1.1 Gợi ý theo luật (từ khoá, bỏ dấu tiếng Việt)

In [ ]:
yc = CH['YEU_CAU']
rb = classify_request(yc, parser.call_chains, parser._priority_order, parser.default_request_type)
sim = pd.DataFrame([{'archetype': a, 'do_khop_tu_khoa': PA._compute_similarity(yc, a, parser.call_chains),
                     'Delta mac dinh %': v['expected_delta_pct'], 'chuoi goi (taxonomy)': v['services']}
                    for a, v in parser.call_chains.items()]).sort_values('do_khop_tu_khoa', ascending=False)
print(f'yeu cau: "{yc}"\nluat goi y: {rb}')
display(sim.reset_index(drop=True))

### 1.2 LLM và sáu guard

| Guard | Kiểm | Khi vi phạm |
|---|---|---|
| **G1** | điểm tiêm tải phải là **gateway** (node vào không có cha) | đổi về gateway trong chuỗi gọi của archetype |
| **G2** | Δ nằm trong [`MIN_DELTA_PCT`, `MAX_DELTA_PCT`] | kẹp vào khoảng |
| **G3** | \|Δ − anchor\| ≤ `MAX_ADJUSTMENT_PCT` | quay về anchor, độ tin cậy LOW |
| **G4** | dịch vụ phải sửa **có thật** trong đồ thị | loại dịch vụ bịa |
| **G5** | độ khớp từ khoá ≥ `SIMILARITY_THRESHOLD` | ép adjustment = 0 |
| **G6** | *Scope gate*: điểm logistic của 4 tín hiệu so với hai ngưỡng hiệu chỉnh conformal | từ chối / cần người xem |

In [ ]:
LLM.nhat_ky.clear()
with quiet():
    parsed = parser.parse(yc)
P = asdict(parsed)
display(pd.Series({k: P[k] for k in ['request_type', 'injection_service', 'template_delta', 'adjustment', 'injection_delta_pct',
                                     'core_services', 'affected_services', 'similarity_score', 'confidence',
                                     'is_out_of_scope', 'needs_human_review', 'scope_gate_score']}, name='ket qua Parser').to_frame())
g = P['scope_gate_score']
vung = 'TU CHOI' if g >= PA._SCOPE_GATE_CONFORMAL_TAU_REFUSE else ('CAN NGUOI XEM' if g >= PA._SCOPE_GATE_CONFORMAL_TAU_PASS else 'CHO QUA')
print(f"G6 scope gate: diem = {g:.3f} | TAU_PASS = {PA._SCOPE_GATE_CONFORMAL_TAU_PASS:.3f} | TAU_REFUSE = "
      f"{PA._SCOPE_GATE_CONFORMAL_TAU_REFUSE:.3f} -> {vung}")
print('dac trung tho cua G6:', P['scope_gate_raw_features'])
print('\nLY DO (ghi cua guard duoc noi vao):\n ', P['reasoning'])

In [ ]:
# Prompt THAT gui cho LLM va cau tra loi nhan ve (xem de hieu LLM duoc phep lam gi)
if LLM.nhat_ky:
    nk = LLM.nhat_ky[-1]
    print(nk['prompt'][:3000] + ('\n...(cat bot)' if len(nk['prompt']) > 3000 else ''))
    print('\n========== LLM TRA LOI ==========\n', nk['tra_loi'][:1500])

## 2. BƯỚC 2 — ArchitectureAgent: vùng ảnh hưởng

Với mỗi dịch vụ Parser trả về, quét đồ thị phụ thuộc để biết **ai gọi nó** (phải báo khi đổi API) và **nó gọi ai**
(phải kiểm tải). Đây là `map_impact_node` của orchestrator.

In [ ]:
impact = {}
for s in set(P['core_services']) | set(P['affected_services']):
    if s in arch.graph:
        impact[s] = {'api_consumers_to_notify': arch.get_upstream_dependencies(s),
                     'downstream_services_to_check': arch.get_downstream_dependencies(s)}
display(pd.DataFrame(impact).T)

## 3. BƯỚC 3 — TẦNG 1: dự báo tài nguyên bằng Global Causal DAG

`CapacityAgent` huấn luyện DAG 28 node trên RCAEval RE2-SS (chi tiết: notebook `01`, Phần A), rồi can thiệp
`do(gateway_workload = nền × (1 + Δ))`. Núm chỉnh: `MAU_CANH_HOC` (bỏ cạnh học để thấy khác biệt), `DELTA_TANG1`.

**Đầu ra là dự báo tài nguyên, KHÔNG phải phán quyết SLO.** Trạng thái SAFE/WARNING/CRITICAL dựa trên % biên dự phòng
so với mức cao nhất đã quan sát (P99 của node).

In [ ]:
_mau = [t for t in CA.DEFAULT_EDGE_TEMPLATES if (not t.learned) or t.name in CH['MAU_CANH_HOC']]
_goc = CA.DEFAULT_EDGE_TEMPLATES
CA.DEFAULT_EDGE_TEMPLATES = _mau            # chi trong bo nho; khoi phuc ngay sau khi huan luyen
try:
    with quiet():
        cap = CA.CapacityAgent(llm=_inner)
finally:
    CA.DEFAULT_EDGE_TEMPLATES = _goc
print(f"DAG: {cap.dag_graph.number_of_nodes()} node, {cap.dag_graph.number_of_edges()} canh | canh hoc: {cap._last_edge_build['learned_edges']}")

p1 = dict(P)
if CH['DELTA_TANG1'] is not None:
    p1['injection_delta_pct'] = float(CH['DELTA_TANG1'])
with quiet():
    ca = cap.assess_capacity(p1, impact)
print(f"Tang 1 -> trang thai {ca.status} | node bao hoa: {ca.saturated_services} | tin cay ngoai suy (G7): {ca.ood_confidence} {ca.ood_flagged_nodes}")
display(pd.DataFrame(ca.simulation_result).T[['n_hops', 'cpu_change_pct', 'cpu_confidence', 'mem_change_pct', 'latency_change_pct', 'latency_confidence']])

In [ ]:
hr = pd.DataFrame({(s, m): v for s, d in ca.headroom.items() for m, v in d.items() if isinstance(v, dict)}).T
display(hr[['baseline', 'projected', 'ceiling_p99', 'consumed_pct', 'remaining_pct', 'status']].round(3))
print('Nhan dinh:', ca.expert_assessment[:600])
print('Khuyen nghi:', ca.recommendations)

## 4. BƯỚC 4 — TẦNG 2: phán quyết SLO ở tải đỉnh

Bộ dự đoán khả thi dựng **từ cấu hình ở mục 0** (khác `assess()` chính thức, vốn luôn dùng bản đóng băng):

1. Cơ chế `ρ, α, β` — fit từ `SS-TRAIN` với `TRAIN_MAX_RPS` (mặc định trùng bản đóng băng).
2. Trần CPU `C_s = 100 × core` — từ `TRAN_CPU`.
3. Ngưỡng `u*` — `U_STAR`.
4. Tham số P2 `c_s`, `x` — `THAM_SO_P2`.
5. Can thiệp: archetype và Δ **lấy từ Parser** (bước 1); `k` từ `K`; chuỗi gọi từ `CHAIN`.

In [ ]:
P2F = json.load(open(FROZEN / 'predictions_frozen_RE2_P2_prosp2.json', encoding='utf-8'))
limits = json.load(open(ROOT / 'deploy/sockshop/limits.json', encoding='utf-8'))['configs']
KDOC = {}
for f in ['k_measured_v3.json', 'k_measured_v4new.json']:
    KDOC.update(json.load(open(FROZEN / f, encoding='utf-8'))['features'])

def co_che(train_max):
    if train_max == P2F['params']['train_max_rps']:
        return P2F['mechanism']
    return FP.fit_mechanism(FP.select_train(FP.load_runs(str(RAW / 'SS-TRAIN')), train_max))

def tran_cpu(cfg):
    c = dict(limits[cfg]) if isinstance(cfg, str) else dict(cfg)
    c = {s: v for s, v in c.items() if not s.startswith('_')}
    return {s: float(c.get(s, 12.0)) for s in FP.SERVICES}

def u_star_phien(ramp_root='SS-PROSP2', cores=None):
    vals = []
    for sp in sorted((RAW / ramp_root / 'ramp_base').glob('run*/steps.json')):
        lo = json.load(open(sp, encoding='utf-8'))['breakpoint']['lo']
        d = pd.read_csv(sp.parent / 'simple_metrics.csv'); d = d[(d.gt_step_warm == 0) & (d.gt_target_rps == lo)]
        vals.append(max(d[f'{s}_cpu'].mean() / (100 * cores[s]) for s in FP.SCORED))
    return float(np.median(vals))

def dung_bo_du_doan(train_max=None, tran=None, u_star=None, p2=None, margin=None):
    """dung FeasibilityPredictor tu cau hinh; tham so None -> lay tu CAU_HINH"""
    train_max = CH['TRAIN_MAX_RPS'] if train_max is None else train_max
    cores = tran_cpu(CH['TRAN_CPU'] if tran is None else tran)
    us = CH['U_STAR'] if u_star is None else u_star
    us = P2F['params']['u_star'] if us == 'dong_bang' else (u_star_phien(cores=cores) if us == 'phien_tien_cuu' else float(us))
    p2 = CH['THAM_SO_P2'] if p2 is None else p2
    p2 = {'c': P2F['p2_params']['c'], 'x': P2F['p2_params']['x']} if p2 == 'dong_bang' else p2
    return FP.FeasibilityPredictor(co_che(train_max), cores, us, margin=CH['U_MARGIN'] if margin is None else margin,
                                   feature_cost=p2)

def k_cho(archetype, k_cfg):
    if k_cfg is None: return None
    if isinstance(k_cfg, dict): return k_cfg
    if k_cfg.startswith('probe:'): return KDOC[k_cfg.split(':', 1)[1]]['measured_per_use']
    ung_vien = [f for f in KDOC if FP.FEATURE_ARCHETYPE.get(f) == archetype]      # 'probe': tinh nang do duoc cung archetype
    return KDOC[ung_vien[0]]['measured_per_use'] if ung_vien else None

def chain_cho(archetype, chain_cfg, k_cfg):
    if chain_cfg == 'taxonomy': return None
    if isinstance(chain_cfg, (list, tuple)): return list(chain_cfg)
    f = k_cfg.split(':', 1)[1] if isinstance(k_cfg, str) and ':' in k_cfg else next((f for f in KDOC if FP.FEATURE_ARCHETYPE.get(f) == archetype), None)
    return KDOC[f]['chain_measured'] if f else None

PRED = dung_bo_du_doan()
ARCH = P['request_type']
SCALE = P['injection_delta_pct'] / SOCKSHOP_CALL_CHAINS[ARCH]['expected_delta_pct']
KK, CHN = k_cho(ARCH, CH['K']), chain_cho(ARCH, CH['CHAIN'], CH['K'])
KW = dict(mode=CH['CHE_DO'], feature=ARCH, scale=SCALE, k=KK, chain=CHN)
print(f"u* = {PRED.u_star:.4f} | archetype {ARCH} | Delta = {P['injection_delta_pct']}% (scale {SCALE:.2f}) | che do {CH['CHE_DO']}"
      f"{' + k do duoc (P3)' if KK else ''} | chain {CHN or 'taxonomy'}")
print('k =', KK)

In [ ]:
if P['is_out_of_scope']:
    print('G6 TU CHOI yeu cau nay -> he thong KHONG dua ra phan quyet dinh luong.')
elif CH['L_PEAK'] is None:
    print('L_PEAK = None -> chi chay Tang 1, khong phan quyet SLO.')
else:
    print(PRED.explain_text(CH['L_PEAK'], **KW))
    R, nut = PRED.breakpoint(**KW)
    v = PRED.verdict(CH['L_PEAK'], **KW)
    cong = PRED.cap.get(nut, 0) / 100 < CH['QUOTA_MIN_CORES']
    ket_luan = 'UNDECIDED (cong throttling)' if cong else v['verdict']
    if CH['N_BOOTSTRAP']:
        tr = FP.select_train(FP.load_runs(str(RAW / 'SS-TRAIN')), CH['TRAIN_MAX_RPS']); rng = np.random.RandomState(0); rs = []
        for _ in range(CH['N_BOOTSTRAP']):
            Pb = FP.FeasibilityPredictor(FP.fit_mechanism(tr.iloc[rng.randint(0, len(tr), len(tr))]),
                                         tran_cpu(CH['TRAN_CPU']), PRED.u_star, feature_cost=PRED.feature_cost)
            rs.append(Pb.breakpoint(**KW)[0])
        ci = f' | KTC 95% bootstrap [{np.percentile(rs, 2.5):.1f}, {np.percentile(rs, 97.5):.1f}]'
    else:
        ci = ''
    print(f"\n===> PHAN QUYET o {CH['L_PEAK']} req/s: {ket_luan} | diem gay R* = {R:.1f} req/s (nut nghen {nut}){ci}")
    if P['needs_human_review']:
        print('     (G6: yeu cau nam o vung CAN NGUOI XEM -- phan quyet chi mang tinh tham khao)')

In [ ]:
Ls = np.linspace(20, 300, 80)
fig, ax = plt.subplots(figsize=(8, 3.6))
for s in FP.SCORED:
    ax.plot(Ls, [PRED.utilization(L, **KW)[s] for L in Ls], label=s)
ax.axhline(PRED.u_star, color='red', ls=':', label=f'u* = {PRED.u_star:.3f}')
ax.axhline(PRED.u_star * (1 - PRED.margin), color='orange', ls=':', label='bat dau MARGINAL')
if CH['L_PEAK']: ax.axvline(CH['L_PEAK'], color='k', ls='--', label='L_PEAK')
ax.set_xlabel('tai nen L (req/s)'); ax.set_ylabel('u = CPU / tran'); ax.legend(fontsize=7, ncol=3)
ax.set_title(f'{ARCH}: muc su dung du doan tung node'); plt.tight_layout(); plt.show()

## 5. So với `assess()` chính thức

Đây là đúng hàm người dùng gọi (`src/agents/orchestrator.py`). Nó **luôn** dùng bản đóng băng và LLM của orchestrator
(offline nếu chưa có API key) — nên nếu bạn đã đổi cấu hình, hai kết quả sẽ khác nhau: đó chính là tác động của thay đổi.

In [ ]:
with quiet():
    from agents import orchestrator as ORCH
    chinh_thuc = ORCH.assess(CH['YEU_CAU'], L_peak=CH['L_PEAK'])
sl = chinh_thuc['slo']
f = sl['feasibility'] if sl else None
so_sanh = pd.DataFrame({
    'assess() chinh thuc': {'archetype': (sl or chinh_thuc['forecast'])['parsed_requirement']['request_type'],
                            'Delta %': (sl or chinh_thuc['forecast'])['parsed_requirement']['injection_delta_pct'],
                            'Tang 1': chinh_thuc['forecast']['capacity_assessment']['status'],
                            'phan quyet': chinh_thuc['verdict'], 'R*': getattr(f, 'breakpoint_rps', None),
                            'mo hinh': getattr(f, 'model', None)},
    'notebook (cau hinh cua ban)': {'archetype': ARCH, 'Delta %': P['injection_delta_pct'], 'Tang 1': ca.status,
                                    'phan quyet': (ket_luan if CH['L_PEAK'] and not P['is_out_of_scope'] else None),
                                    'R*': round(R, 1) if CH['L_PEAK'] and not P['is_out_of_scope'] else None,
                                    'mo hinh': CH['CHE_DO'] + (' + k' if KK else '')}})
display(so_sanh)

Lưu ý: `assess()` chính thức **không truyền `k`** trừ khi bạn đưa vào, nên nó chạy P2 (k = 1). Notebook mặc định
`K='probe'` (P3) — đó là khác biệt duy nhất ở cấu hình mặc định.

---
## 6. 🧪 Bàn thực nghiệm — thử biến thể mô hình trên n = 10 tính năng tiến cứu

Cách dùng: viết một hàm `du_doan(tinh_nang) -> điểm gãy (req/s)`, rồi gọi `danh_gia(du_doan, 'tên')`. Hàm tự so với
đáp án đo được, với P3 đóng băng, với hai mốc nền, với sàn nhiễu — và ghi vào **bảng xếp hạng**.

Quy ước dấu: sai số **dương** = đoán điểm gãy **muộn** hơn thực tế = **khả thi giả** (nguy hiểm).

In [ ]:
FEATS = ['login', 'register', 'wishlist', 'catsearch', 'account', 'preview', 'orderhist', 'related', 'reorder', 'orderfull']
LO, HI = {}, {}
for sp in sorted((RAW / 'SS-PROSP2').glob('ramp_*/run*/steps.json')):
    j = json.load(open(sp, encoding='utf-8')); f = j.get('feature', 'base')
    if f != 'base' and (f not in FEATS or round(j['feature_pct'] / HARNESS.FEATURES[f]['anchor'], 3) != 1.0):
        continue
    LO.setdefault(f, []).append(j['breakpoint']['lo']); HI.setdefault(f, []).append(j['breakpoint']['hi'] or 1.25 * j['breakpoint']['lo'])
DAP_AN = {f: float(np.median(LO[f])) for f in FEATS}
P3_DB = {f: next(p['breakpoint_rps'] for p in json.load(open(FROZEN / f'predictions_frozen_RE2_P3_prosp_{f}.json', encoding='utf-8'))['predictions'] if p['scale'] == 1.0) for f in FEATS}
SAN_NHIEU = float(np.median([100 * (max(v) - min(v)) / np.median(v) for v in LO.values()]))
BANG_XEP_HANG = []

def danh_gia(du_doan, ten, in_chi_tiet=True):
    from scipy import stats
    pred = {f: float(du_doan(f)) for f in FEATS}
    e = pd.Series({f: 100 * (pred[f] - DAP_AN[f]) / DAP_AN[f] for f in FEATS})
    e3 = pd.Series({f: 100 * (P3_DB[f] - DAP_AN[f]) / DAP_AN[f] for f in FEATS})
    eo = pd.Series({f: 100 * (np.median(list(DAP_AN.values())) - DAP_AN[f]) / DAP_AN[f] for f in FEATS})
    nguy = [f for f in FEATS if pred[f] > np.median(HI[f])]
    giong_p3 = all(abs(pred[f] - P3_DB[f]) <= 0.05 for f in FEATS)        # P3 trong file lam tron 0,1 req/s
    p3 = 1.0 if giong_p3 else stats.wilcoxon(e.abs(), e3.abs()).pvalue
    po = stats.wilcoxon(e.abs(), eo.abs()).pvalue
    dong = {'bien the': ten, '|sai so| TB %': round(e.abs().mean(), 1), 'trung vi %': round(e.abs().median(), 1),
            'xau nhat %': round(e.abs().max(), 1), 'kha thi gia': len(nguy), 'p vs P3': round(p3, 3), 'p vs const_oracle': round(po, 3)}
    BANG_XEP_HANG[:] = [r for r in BANG_XEP_HANG if r['bien the'] != ten] + [dong]
    if in_chi_tiet:
        display(pd.DataFrame({'dap an': DAP_AN, 'du doan': pred, 'sai so %': e.round(1), 'P3 dong bang': P3_DB,
                              'sai so P3 %': e3.round(1)}).round(1))
        print(f"{ten}: |sai so| TB {dong['|sai so| TB %']}% (P3 {e3.abs().mean():.1f}%, san nhieu {SAN_NHIEU:.1f}%) | "
              f"kha thi gia {len(nguy)} {nguy} | Wilcoxon p vs P3 = {p3:.3f}")
    return dong

def xem_bang_xep_hang():
    d = pd.DataFrame(BANG_XEP_HANG).sort_values('|sai so| TB %').reset_index(drop=True)
    display(d); print(f'san nhieu phep do = {SAN_NHIEU:.1f}%  -- chenh lech nho hon muc nay giua hai bien the la CHUA phan biet duoc')

def diem_gay(f, **ghi_de):
    """diem gay cua tinh nang f (ten trong FEATS) theo bo du doan dung tu CAU_HINH, co the ghi de tung tham so.
    ghi_de: u_star, tran, p2, train_max, margin (bo du doan); mode, chain ('taxonomy'|'probe'|list), k ('probe'|None|dict), delta_nhan"""
    bd = dung_bo_du_doan(**{k: ghi_de[k] for k in ['u_star', 'tran', 'p2', 'train_max', 'margin'] if k in ghi_de})
    kcfg = ghi_de.get('k', 'probe')
    k = KDOC[f]['measured_per_use'] if kcfg == 'probe' else kcfg
    ch = ghi_de.get('chain', 'taxonomy')
    ch = None if ch == 'taxonomy' else (KDOC[f]['chain_measured'] if ch == 'probe' else ch)
    return bd.breakpoint(mode=ghi_de.get('mode', 'P2'), feature=f, scale=ghi_de.get('delta_nhan', 1.0), k=k, chain=ch)[0]

### E0 — kiểm chứng bàn: tái lập P3 đóng băng (phải cho **đúng** 9,5%)

In [ ]:
danh_gia(lambda f: diem_gay(f, u_star='dong_bang', p2='dong_bang', train_max=150, tran='RE2'), 'E0 P3 dong bang (tai lap)');

### E1 — u* hiệu chỉnh theo phiên đo tiến cứu (nguồn sai số chính theo chẩn đoán)
⚠️ Dùng ramp `base` của chính phiên tiến cứu → đây là **hồi cứu**, chỉ để biết hướng cải tiến có đáng làm không.

In [ ]:
danh_gia(lambda f: diem_gay(f, u_star='phien_tien_cuu'), 'E1 u* theo phien');

### E2 — P3 + chuỗi gọi đo được (thay chuỗi taxonomy)

In [ ]:
danh_gia(lambda f: diem_gay(f, chain='probe'), 'E2 P3 + chain do duoc');

### E3 — quét u*: sai số thay đổi thế nào theo ngưỡng?
Đường cong này cho thấy **độ nhạy** — và vì sao chọn u* bằng cách nhìn đường này là *gian lận* (fit trên đáp án).
Muốn chọn u* trung thực: dùng LOFO ở E5.

In [ ]:
qu = np.round(np.arange(0.66, 0.92, 0.02), 2)
curve = [danh_gia(lambda f, u=u: diem_gay(f, u_star=u), f'quet u*={u}', in_chi_tiet=False) for u in qu]
BANG_XEP_HANG[:] = [r for r in BANG_XEP_HANG if not r['bien the'].startswith('quet u*')]    # khong dua vao bang xep hang
cv = pd.DataFrame(curve)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(qu, cv['|sai so| TB %'], 'o-'); ax[0].axhline(SAN_NHIEU, color='gray', ls=':', label='san nhieu')
ax[0].axvline(P2F['params']['u_star'], color='red', ls=':', label='u* dong bang'); ax[0].set_xlabel('u*'); ax[0].set_ylabel('|sai so| TB %'); ax[0].legend(fontsize=7)
ax[1].bar(qu, cv['kha thi gia'], width=.015); ax[1].set_xlabel('u*'); ax[1].set_ylabel('so o kha thi gia')
plt.tight_layout(); plt.show()

### E4 — độ nhạy theo Δ (hạn chế M5: Δ thật chưa bao giờ bị kiểm vì harness bơm đúng Δ)
Nếu ngoài đời Δ ước lượng sai ±50%, điểm gãy dự đoán lệch bao nhiêu?

In [ ]:
nhan = [0.5, 0.75, 1.0, 1.25, 1.5]
dn = pd.DataFrame({f: [diem_gay(f, delta_nhan=m) for m in nhan] for f in FEATS}, index=[f'Delta x{m}' for m in nhan]).T
display((100 * (dn.div(dn['Delta x1.0'], axis=0) - 1)).round(1).rename(columns=lambda c: c + ' (% doi R*)'))

### E5 — mẫu LOFO: chọn tham số trung thực (bỏ một tính năng)
Với mỗi tính năng *f*: chọn u* tốt nhất trên **9 tính năng còn lại**, rồi dự đoán *f* bằng u* đó. Kết quả là ước lượng
trung thực của "u* hiệu chỉnh lại" — thay mẫu này bằng bất kỳ tham số nào bạn muốn fit.

In [ ]:
LUOI_U = np.round(np.arange(0.60, 0.95, 0.01), 2)
bang_R = {u: {f: diem_gay(f, u_star=u) for f in FEATS} for u in LUOI_U}          # tinh truoc cho nhanh
def lofo_u(f_giu):
    con_lai = [g for g in FEATS if g != f_giu]
    loi = {u: np.mean([abs(bang_R[u][g] - DAP_AN[g]) / DAP_AN[g] for g in con_lai]) for u in LUOI_U}
    u_tot = min(loi, key=loi.get)
    return bang_R[u_tot][f_giu], u_tot
chon = {f: lofo_u(f)[1] for f in FEATS}
print('u* LOFO chon cho tung tinh nang bi giu lai:', chon)
danh_gia(lambda f: lofo_u(f)[0], 'E5 u* chon bang LOFO');

### Đọc E1–E5 (kết quả với cấu hình mặc định, chạy 28/09/2026)

| Biến thể | \|sai số\| TB | So với P3 (9,5%) |
|---|---|---|
| E1 u* theo phiên (0,803) | 9,6% | **không** tốt hơn: login/register/wishlist/orderhist bớt muộn, nhưng catsearch/account/related/reorder/orderfull thành quá sớm |
| E2 chain đo được | 8,6% | tốt hơn chút, p = 0,54 — chưa phân biệt được |
| E5 u* chọn bằng LOFO | 10,9% | **kém hơn** — chọn u* trung thực không cải thiện gì |

**Kết luận:** sai số **không phải một độ lệch chung** mà hạ một u* là sửa được. Các tính năng lệch theo **hai chiều ngược
nhau** — tuyến ghi phiên (login/register) vỡ sớm hơn dự đoán, còn đọc catalogue/đơn hàng vỡ muộn hơn. Muốn tiến bộ phải
có ngưỡng hoặc cơ chế **phụ thuộc loại tính năng** (vd. mô hình độ trễ/hàng đợi theo từng lớp request), không phải chỉnh
một hằng số. Mọi chênh lệch ở đây đều nhỏ hơn sàn nhiễu 21,4% → cần lưới đo mịn hơn trước khi xếp hạng chắc chắn.

*(Nếu bạn đổi cấu hình ở mục 0, các con số trên sẽ khác — bảng này ghi kết quả mặc định để đối chiếu.)*

### ✍️ Ô của bạn — viết biến thể mới ở đây

Ví dụ: đổi trần CPU front-end lên 0,6 core, hoặc tự viết một hàm dự đoán hoàn toàn mới (miễn trả về req/s):

In [ ]:
# vi du 1: tran front-end 0.6 core (chi trong bo nho)
tran_moi = dict(limits['RE2']); tran_moi['front-end'] = 0.6
danh_gia(lambda f: diem_gay(f, tran=tran_moi), 'VD tran front-end 0.6', in_chi_tiet=False)

# vi du 2: ham tu viet -- P3 nhung u* giam 5% cho tinh nang quat-ra nhieu (Sum k >= 4)
def bien_the_cua_toi(f):
    sk = sum(v for s, v in KDOC[f]['measured_per_use'].items() if s != 'front-end')
    return diem_gay(f, u_star=P2F['params']['u_star'] * (0.95 if sk >= 4 else 1.0))
danh_gia(bien_the_cua_toi, 'VD u* giam cho quat-ra', in_chi_tiet=False)

xem_bang_xep_hang()

## 7. Khi một biến thể đáng giữ — đưa vào hệ thống thật

| Bước | Việc | Vì sao |
|---|---|---|
| 1 | Kiểm lại bằng **LOFO** (mẫu E5) nếu biến thể có tham số fit | n = 10: fit thẳng trên đáp án luôn "thắng" |
| 2 | Thêm thành **chế độ mới** (vd `mode='P4'`) trong `src/scm/feasibility_predictor.py`; **không sửa** P0–P3 | bản đóng băng phải tái lập đúng từng bit |
| 3 | Viết test trong `tests/`, chạy `pytest tests -q` | bắt lỗi hồi quy |
| 4 | Đóng băng bằng `freeze_predictions.py` / `freeze_p3_prospective.py`, rồi **commit và push ngay** | hash chứng minh nội dung, **git trên remote** chứng minh thời điểm |
| 5 | Agent mù cài **tính năng mới** → probe → ramp → chấm | chỉ bước này mới là **tiến cứu** |

Chỗ cắm trong hệ thống: `NewFeatureFeasibilityAgent.assess(..., model='P4')` → `orchestrator.assess()`.